# M15 L15: Staff-Level System Design for AI-Native Data Systems

**Build a capacity estimator + design-doc template: 4 sizing knobs, 3 scenarios, 17 tests.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

When a grown-up asks "how big a computer do I need?", they don't draw a picture -- they do **math**. Five numbers in, four numbers out: how many computers, how much storage, how much it costs per month, and how much it can be down before users get mad.

The math is simple, but the answers are not. At 10,000 queries per second, we need 120 computers and a $260K monthly bill -- and 99% of that bill is the LLM (the smart part that reads the docs). The "how much can it be down" answer is the **error budget**: if you promise 99.9% uptime, you only have **43 minutes per month** of allowed downtime. One bug, one bad deploy, and your budget is gone.

This is what "staff-level" means: the numbers add up, and you can explain them.

## The 5 things to know

1. **Four knobs of any design**: capacity (instances/GB/$), SLOs (latency/availability/freshness/recall@k), sharding (key + skew), cost (compute/storage/LLM/egress).
2. **Capacity formula**: `n_instances = ceil(QPS / 100 * 1.2)` -- 1 box = 100 QPS, 20% headroom.
3. **Error budget**: `(1 - availability) * window_in_minutes`. 99.9% over a month = 43 min of allowed downtime.
4. **LLM cost dominates**: at $0.001 per call, 1B calls/month = $1M. Cache and model routing are how you control it.
5. **Shard skew is the most common production failure**: per-tenant (10x), per-date (1.5x), per-language (20x).

In [1]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm15-staff-design' / 'L15-staff-design.py'
_spec = importlib.util.spec_from_file_location('L15', _p)
_L15 = importlib.util.module_from_spec(_spec)
sys.modules['L15'] = _L15
_spec.loader.exec_module(_L15)

print(f'Setup OK. Lesson module: {_L15.LESSON_NUMBER}')

Setup OK. Lesson module: M15.L15


In [2]:
# Run the 17 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L15, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 29 tests...

  PASS  test_capacity_plan_basic
  PASS  test_capacity_plan_cost_scales_with_regions
  PASS  test_capacity_plan_latency_increases_with_complexity
  PASS  test_capacity_plan_latency_under_budget
  PASS  test_capacity_plan_storage_inversely_with_regions
  PASS  test_capacity_plan_storage_scales_with_qps
  PASS  test_cost_model_share_sums_to_one
  PASS  test_cost_model_share_sums_to_one_v2
  PASS  test_cost_model_total
  PASS  test_design_doc_renders
  PASS  test_error_budget_four_nines
  PASS  test_error_budget_one_nine_is_large
  PASS  test_error_budget_three_nines
  PASS  test_plan_shards_by_tenant_with_seed_default_42
  PASS  test_plan_shards_by_tenant_with_seed_other_value
  PASS  test_run_demo_seed_42_is_deterministic
  PASS  test_scenario_10m_users_expensive
  PASS  test_scenario_1k_users_cheap
  PASS  test_scenario_rag_10m_users_is_expensive
  PASS  test_scenario_rag_1k_cheaper_than_10m
  PASS  test_shard_plan_by_date_balanced
  PASS  test_shard_plan_by_langua

In [3]:
# === Section 1: capacity plan (1000 QPS, 99.9%, 3 regions) ===
inputs = _L15.CapacityInputs(
    qps=1000, p99_latency_ms=400, availability_target=0.999, region_count=3, model='gpt-4o-mini',
)
out = _L15.capacity_plan(inputs)
print(f'  n_instances:        {out.n_instances}')
print(f'  storage_gb:         {out.storage_gb}')
print(f'  monthly_cost_usd:   ${out.monthly_cost_usd:,.2f}')
print(f'  error_budget (min): {out.error_budget_per_month}')

  n_instances:        12
  storage_gb:         3.33
  monthly_cost_usd:   $7,850,132.20
  error_budget (min): 43.2


In [4]:
# === Section 2: SLO + error budget ===
slo = _L15.SLO(latency_p99=400, availability=0.999, freshness=300, recall_at_k=0.9)
print(f'SLO: p99={slo.latency_p99}ms avail={slo.availability} fresh={slo.freshness}s r@k={slo.recall_at_k}')
obs_ok = {'latency_p99': 380, 'availability': 0.9995, 'freshness': 250, 'recall_at_k': 0.92}
obs_bad = {'latency_p99': 600, 'availability': 0.99, 'freshness': 250, 'recall_at_k': 0.92}
print(f'  observed OK:   passes={slo.passes(obs_ok)}')
print(f'  observed BAD:  passes={slo.passes(obs_bad)}')
budget = _L15.ErrorBudget(availability=0.999, window_minutes=_L15.MINUTES_PER_MONTH)
print(f'  error budget: {budget.minutes:.2f} min/month, {budget.nines:.2f} nines, remaining={budget.remaining(20):.2f}')

SLO: p99=400ms avail=0.999 fresh=300s r@k=0.9
  observed OK:   passes=True
  observed BAD:  passes=False
  error budget: 43.20 min/month, 3.00 nines, remaining=23.20


In [5]:
# === Section 3: shard planning ===
plans = [
    _L15.plan_shards_by_tenant(n_tenants=100),
    _L15.plan_shards_by_date(n_days=365, shards_per_day=1),
    _L15.plan_shards_by_language(n_languages=12),
]
print(f'{"key":<10} {"n_shards":>9} {"skew":>7}  balanced?')
for p in plans:
    print(f'  {p.key:<10} {p.n_shards:>9} {p.skew_ratio:>7.1f}  {p.is_balanced()}')

key         n_shards    skew  balanced?
  tenant_id        100    10.0  False
  date             365     1.5  True
  language          12    20.0  False


In [6]:
# === Section 4: cost model ===
cost = _L15.build_cost_model(n_instances=12, storage_gb=10, qps=1000)
print(f'\n{"component":<24} {"$/month":>12}')
for c in cost.components:
    print(f'  {c}')
print(f'  {"TOTAL":<24} ${cost.total:>10,.2f}')
print(f'  llm share: {cost.share("llm"):.1%}    compute share: {cost.share("compute"):.1%}')


component                     $/month
  compute                  $  1,382.40 / month
  storage                  $      1.00 / month
  llm                      $2,592,000.00 / month
  egress                   $ 23,328.00 / month
  TOTAL                    $2,616,711.40
  llm share: 99.1%    compute share: 0.1%


In [7]:
# === Section 5: three capacity scenarios ===
print('\nThree scenarios:')
print(f'  {"scenario":<28}  {"instances":>10}  {"$/month":>12}  {"budget_min":>10}')
for scenario in (_L15.scenario_rag_1k_users(), _L15.scenario_rag_100k_users(), _L15.scenario_rag_10m_users()):
    out = scenario['outputs']
    cost = scenario['cost']
    print(f'  {scenario["name"]:<28}  {out.n_instances:>10}  ${cost.total:>11,.2f}  {out.error_budget_per_month:>10.2f}')


Three scenarios:
  scenario                       instances       $/month  budget_min
  1K users (single region)               1  $  26,268.49      432.00
  100K users (3 regions)                12  $2,616,710.73       43.20
  10M users (5 regions)               1200  $261,671,060.00        4.32


In [8]:
# === Section 6: design doc example ===
doc = _L15.DesignDoc(
    problem='Build a RAG system for 100K internal users answering questions about HR/Finance/Ops policies.',
    requirements=[
        'Functional: ingest docs, chunk, embed, retrieve, answer with citations',
        'Non-functional: p99 < 400ms, availability 99.9%, freshness < 5min, recall@10 >= 0.9',
        'Constraint: must run on existing 3-region infra; budget $10K/month',
    ],
    architecture=(
        'Ingest: S3 -> chunker -> embedder -> vector DB (3 regions, 1 shard per language). '
        'Query: API gateway -> retriever (top-K=10) -> reranker -> LLM (gpt-4o-mini) -> answer + citations.'
    ),
    tradeoffs=[
        'Per-language shards: skew=20x (English dominates) but enables lang-specific tokenizers',
        'Rerank on every query: +100ms latency, +20% recall -- chose recall over latency',
        'No cross-region replication of the index: cheaper but US-EU queries are slower',
    ],
    ops=[
        'SLOs: p99 400ms, 99.9% availability, freshness 300s, recall@10 0.9',
        'Alerts: p99 > 500ms for 5min, error rate > 1% for 5min, freshness > 600s',
        'On-call: 1 IC + 1 backup; runbook for index rebuild, LLM fallback, region failover',
    ],
)
print(doc.render())

  DESIGN DOC

PROBLEM:
  Build a RAG system for 100K internal users answering questions about HR/Finance/Ops policies.

REQUIREMENTS:
  - Functional: ingest docs, chunk, embed, retrieve, answer with citations
  - Non-functional: p99 < 400ms, availability 99.9%, freshness < 5min, recall@10 >= 0.9
  - Constraint: must run on existing 3-region infra; budget $10K/month

ARCHITECTURE:
  Ingest: S3 -> chunker -> embedder -> vector DB (3 regions, 1 shard per language). Query: API gateway -> retriever (top-K=10) -> reranker -> LLM (gpt-4o-mini) -> answer + citations.

TRADEOFFS:
  - Per-language shards: skew=20x (English dominates) but enables lang-specific tokenizers
  - Rerank on every query: +100ms latency, +20% recall -- chose recall over latency
  - No cross-region replication of the index: cheaper but US-EU queries are slower

OPS:
  - SLOs: p99 400ms, 99.9% availability, freshness 300s, recall@10 0.9
  - Alerts: p99 > 500ms for 5min, error rate > 1% for 5min, freshness > 600s
  - On-cal

## Architecture (Mermaid)

```mermaid
flowchart TB
    subgraph Inputs
        QPS[QPS target]
        LAT[p99 latency]
        AVAIL[availability SLO]
        REG[region count]
    end

    subgraph "Capacity Plan"
        INST[n_instances]
        STO[storage_gb]
        COST[$/month]
    end

    subgraph "SLO + Error Budget"
        SLO[SLO: p99, avail, fresh, r@k]
        EB[Error budget: N min/month]
    end

    subgraph "Sharding"
        KEY[shard key: tenant / date / language]
        SKEW[skew ratio]
    end

    subgraph "Cost Model"
        CMP[compute]
        STG[storage]
        LLM[llm calls]
        EGR[egress]
    end

    QPS --> INST
    QPS --> STO
    AVAIL --> EB
    REG --> COST
    INST --> CMP
    STO --> STG
    QPS --> LLM
    REG --> EGR
    COST --> SLO
    EB --> SLO
    KEY --> SKEW
```

**Read as:** inputs feed four knobs (capacity, SLO, sharding, cost); the numbers cross-link.

## Reflect (10 min)

1. In the three scenarios (1K / 100K / 10M users), which cost component grows fastest: compute, storage, LLM, or egress? Why?
2. Why is per-language sharding more skewed than per-tenant sharding? What would a composite (tenant + language) shard plan look like?
3. If your SLO is 99.99% and you have a 4-minute outage, is the SLO busted? What about 8 minutes? Why does it matter what *window* you measure over?
4. The capacity model assumes 100 QPS per instance. In what situation would this be wildly wrong? (Hint: think about LLM latency variance.)
5. (Stretch) Add a `cost_optimization_plan()` that, given a budget cap, suggests where to cut.

---

**Read next:**
- `lessons/m16-fullstack/L16-fullstack.ipynb` -- tying modules M1-M15 into a deployable system
- `lessons/m17-product-metrics/L17-product-metrics.ipynb` -- how product metrics relate to SLOs
- `SYLLABUS.md` -- the 18 modules and 21-day plan